# N/V 이력의 작은 선형 변환 — M2·M5 개발 비교

## 목적과 구조
기존 `q_N × p_N, q_V × p_V` 대신 **h_N=W_N[p_N;q_N]+b_N, h_V=W_V[p_V;q_V]+b_V**를 사용합니다. 축당 4차원으로 추가 파라미터는 총 48개입니다. ID LightGCN과 모두 하나의 optimizer·추천손실로 공동학습하며 동결·외부 재정렬은 없습니다.

- N/V는 기존 학습구간 historical CLV 구성요소를 그대로 사용합니다. M2에는 q_C가 없습니다.
- 학습은 양성 상품을 제외한 이력(LOO), 평가는 전체 학습이력입니다. 유효하지 않거나 LOO 후 빈 이력은 변환 후에도 0입니다.
- 변환은 **비선형 게이트가 아니라 선형 결합**입니다. N/V 입력의 방향을 학습하지만 고객별 행렬이나 반복구매율 게이트를 학습하는 구조는 아닙니다.
- 변환된 사용자 블록은 단위 정규화하지 않습니다. ID 블록과의 크기 불균형을 자동으로 해결한다고 주장하지 않으며 점수 비중·노름을 기록합니다.
- 기존 L2 계수 1e-3을 새 공유 변환의 weight와 bias에도 적용합니다(배치당 공유 파라미터 제곱합 한 번).
- M5는 **보완 M4** `1+0.5*q_C*(1-RBF_value_fit)`를 유지합니다. 원형 M4와 혼용하지 않습니다.

## 고정 설정과 비용
Dunnhumby 개발구간 684–690, seed 42·43·44, 100 epoch, ID 64차원·2층, rho=0.05, K=1 uniform, binary graph, MIN_ITEM_INTER=1. 최종 test·holdout은 평가하지 않습니다.

시드당 새 M2 / 새 M5 / 상수-q M2 / 상수-q M5 = **4회**, 총 **12회 새 학습**입니다. 48개만 따로 학습하는 것이 아니라 전체 모델을 공동학습합니다. 학습시간은 실측 로그로 추정하세요. 기존 M1·M2·M4·M5는 호환 결과만 재사용하고 없으면 중단합니다.

## 판독 규칙 — 실행 전 고정
주지표는 **전체 가격·구매금액 가중 적중값@10 및 전체 가중 NDCG@10**입니다. M2는 M1과, M5는 M4와 비교하며 두 지표 모두 증가하는지 봅니다. 전체 Recall/NDCG @10·20·50 각각이 matched M1의 99% 이상인지 확인합니다. **시드별 결과와 평균을 별도로** 보고합니다.

기존 M2/M5 대비 및 상수-q 대조도 함께 보고하며, @20/@50·저/중/고CLV·노출 지표는 모두 저장합니다. 상수 대조는 N/V 이력을 유지하므로 **모든 CLV 정보를 제거한 대조가 아닙니다**. 3개 개발 시드로 유의성·일반화·CLV 귀속을 확정하지 않습니다. 성과가 나온 뒤 판정 기준을 바꾸지 않습니다.

## 실행
1–3번 셀은 설치·설정·기존 결과 확인이며 **학습하지 않습니다**. 4번 셀부터 학습합니다. 매 epoch optimizer·난수 상태를 Drive에 저장합니다. 세션이 끊기면 같은 설정으로 재실행해 재개합니다(Colab 세션 자동 재연결 기능은 아님).

진행 중인 H&M 노트북과 별도 런타임에서 사용하세요. 같은 seed를 두 런타임에서 동시에 실행하지 마세요.


In [ ]:
# 1. 설치 / 기존 실행과 분리된 고정 커밋 경로
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, json
from pathlib import Path
SOURCE_COMMIT = 'dd0716cc6a41d08502fc566bf2dc9a111482f022'
REPO = Path('/content/clv-history-linear-nv-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '다른 실험 모듈이 남아 있습니다. 별도 런타임 또는 세션 재시작이 필요합니다.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('고정 실행 코드:', SOURCE_COMMIT)


In [ ]:
# 2. 설정 출력만 — 학습 없음
import torch
import pandas as pd
import lightgcn_clv_history_linear_nv as screen
SEEDS = (42, 43, 44)
cfg = screen.configure(seeds=SEEDS)
print(json.dumps(screen.preflight(cfg), ensure_ascii=False, indent=2))
print('결과 폴더:', cfg.out_dir)
print('기존 결과 폴더:', cfg.reuse_dirs)
assert torch.cuda.is_available(), '새 학습을 위해 GPU 런타임을 선택하세요.'


In [ ]:
# 3. 데이터 준비 + 기존 M1/M2/M4/M5 호환 결과 확인 — 학습 없음
# 누락되면 여기서 중단합니다. 기존 모델을 자동 재학습하지 않습니다.
prepared = screen.prepare(cfg)
anchors = prepared['linear_nv_anchors']
display(pd.DataFrame([{k: a.get(k) for k in ('seed', 'model_id', 'origin', 'source_result')} for a in anchors]))
print('확인 완료. 다음 셀부터 새 모델 4개 ×', len(SEEDS), '시드를 공동학습합니다.')


In [ ]:
# 4. 실제 학습 시작: 매 epoch 체크포인트, 완료 arm 재사용
absolute = screen.run(cfg, prepared=prepared)
print(json.dumps(absolute.attrs['paths'], ensure_ascii=False, indent=2))


In [ ]:
# 5. 요약 표시 + 전체 원본 보존 (화면 5,000줄 잘림 방지)
paths = absolute.attrs['paths']
summary = pd.read_csv(paths['summary'])
reading = pd.read_csv(paths['reading'])
key_metrics = list(screen.base.ACCURACY) + list(screen.PRIMARY) + [
    'price_purchase_amount_weighted_hit@20', 'price_purchase_amount_weighted_hit@50',
    'vndcg@20', 'vndcg@50']
display(summary[summary.metric.isin(key_metrics)].reset_index(drop=True))
display(reading)
print('전체·세그먼트·노출 지표와 불리한 결과도 아래 파일에 생략 없이 저장했습니다.')
for name, path in paths.items():
    print(name, ':', path)
print('요약의 평균 통과는 모든 시드 통과와 다릅니다. 유의성·최종 성과 판정이 아닙니다.')


In [ ]:
# 6. 전체 결과 ZIP 다운로드 (체크포인트는 큰 파일이므로 제외)
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path('/content/m2_m5_linear_nv_results.zip')
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as z:
    for path in paths.values():
        z.write(path, arcname=Path(path).name)
files.download(str(archive))
